In [ ]:
# !pip install folium branca

In [ ]:
import pandas as pd
import numpy as np
import h3
import folium
from pathlib import Path

# ── Cargar parquet y filtrar Milán ──────────────────────────────────────────
df = pd.read_parquet('dataset_h3_multicidad.parquet')
df_mil = df[df['city'] == 'milano'].copy()

# Demanda total por celda (para colorear el mapa)
cell_demand = (
    df_mil.groupby('h3_cell')
    .agg(
        total_trips  = ('target_demanda', 'sum'),
        mean_demand  = ('target_demanda', 'mean'),
        pct_active   = ('target_demanda', lambda x: (x > 0).mean() * 100)
    )
    .reset_index()
)

print(f'Celdas H3 activas en Milán: {len(cell_demand)}')
print(f'Total viajes: {cell_demand["total_trips"].sum():,.0f}')
cell_demand.describe()

In [ ]:
import branca.colormap as cm

# ── Escala de color por demanda total ───────────────────────────────────────
vmin = cell_demand['total_trips'].quantile(0.05)
vmax = cell_demand['total_trips'].quantile(0.95)
colormap = cm.LinearColormap(
    colors=['#f7fbff', '#6baed6', '#08519c'],  # blanco → azul claro → azul oscuro
    vmin=vmin, vmax=vmax,
    caption='Total viajes iniciados (percentiles 5-95)'
)

# ── Crear mapa centrado en Milán ─────────────────────────────────────────────
MILAN_CENTER = [45.4654, 9.1859]

m = folium.Map(
    location=MILAN_CENTER,
    zoom_start=12,
    tiles='CartoDB positron'   # fondo limpio, ideal para ver la rejilla
)

# ── Dibujar cada celda H3 ───────────────────────────────────────────────────
for _, row in cell_demand.iterrows():
    cell = row['h3_cell']
    total = row['total_trips']
    mean_d = row['mean_demand']
    pct = row['pct_active']

    # h3.cell_to_boundary devuelve lista de (lat, lng)
    boundary = h3.cell_to_boundary(cell)   # [(lat,lng), ...]
    polygon_coords = [[lat, lng] for lat, lng in boundary]

    color = colormap(min(total, vmax))

    folium.Polygon(
        locations=polygon_coords,
        color='#2c5282',
        weight=0.8,
        fill=True,
        fill_color=color,
        fill_opacity=0.65,
        tooltip=(
            f"<b>Celda:</b> {cell}<br>"
            f"<b>Total viajes:</b> {int(total)}<br>"
            f"<b>Demanda media/hora:</b> {mean_d:.3f}<br>"
            f"<b>% horas activas:</b> {pct:.1f}%"
        )
    ).add_to(m)

colormap.add_to(m)

# ── Guardar y mostrar ────────────────────────────────────────────────────────
output_path = 'mapa_h3_milan.html'
m.save(output_path)
print(f'Mapa guardado: {output_path}')
m   # muestra el mapa inline en Jupyter